# Notebook 3 — Cluster Analysis & Dimensionality Reduction (PCA + K-Means)

**Duration:** 105 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** apply PCA for dimensionality reduction, interpret principal components, and perform K-Means clustering to identify groups of similar iron complexes.

## Learning objectives

- Apply Principal Component Analysis (PCA) to high-dimensional feature matrices.
- Interpret PCA loadings and visualize data in low-dimensional PCA space.
- Use the Elbow Method and Silhouette Score to select the number of clusters.
- Apply K-Means and characterize clusters by feature statistics.

---

## Part 1 — Theoretical background (25 minutes)

### 1.1 The curse of dimensionality

When working with many features (e.g., dozens of structural descriptors for each iron complex) distances between points become less meaningful, models require exponentially more data to generalize, and visualization beyond 3D is impossible. Dimensionality reduction projects data from a high-dimensional space $\mathbb{R}^d$ to a lower-dimensional space $\mathbb{R}^k$ (where $k \ll d$) while preserving as much information as possible.

### 1.2 Principal Component Analysis (PCA)

PCA finds directions of maximum variance in the data. Given a centered data matrix $X \in \mathbb{R}^{n\times d}$, the covariance matrix is:

$$
C = \frac{1}{n-1} X^\top X
$$

PCA solves the eigenproblem

$$
C \mathbf{v}_i = \lambda_i \mathbf{v}_i, \qquad \lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d
$$

and the principal component projection onto the top-$k$ eigenvectors $V_k = [\mathbf{v}_1,\dots,\mathbf{v}_k]$ is

$$
Z = X V_k
$$

The explained variance ratio for component $i$ is

$$
\mathrm{EVR}_i = \frac{\lambda_i}{\sum_{j=1}^d \lambda_j}
$$

### 1.3 K-Means clustering

K-Means partitions $n$ data points into $K$ clusters by minimizing the within-cluster sum of squares (WCSS):

$$
\mathrm{WCSS} = \sum_{k=1}^K \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2
$$

Algorithm (high level):

1. Initialize $K$ centroids (randomly or with k-means++).
2. Assign each point to the nearest centroid.
3. Update each centroid to be the mean of its assigned points.
4. Repeat steps 2–3 until convergence.

Use the Elbow Method (WCSS vs K) and the Silhouette Score to guide choice of $K$.

---

## Part 2 — Applying PCA (30 minutes)

### 2.1 Data preparation

Load the cleaned dataset produced in Day 2 and select numeric features and the target. We add a defensive note: adjust the target column manually if the heuristic picks an unexpected column.



In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')

REPO_PATH = os.path.expandvars('$SCRATCH/Fe-Redox-GNN')
df = pd.read_csv(os.path.join(REPO_PATH, 'data_cleaned.csv'))

# Identify numeric columns and pick a target heuristically (adjust if needed)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    raise RuntimeError('No numeric columns found in data_cleaned.csv')

TARGET_COL = numeric_cols[-1]  # Adjust manually if this is not the desired target
FEATURE_COLS = [c for c in numeric_cols if c != TARGET_COL]

print(f"Dataset: {df.shape[0]} samples × {df.shape[1]} columns")
print(f"Target (heuristic): {TARGET_COL}")
print(f"Number of features: {len(FEATURE_COLS)}")

display(df.head())




### 2.2 Standardize features (critical for PCA)

PCA is sensitive to feature scales; standardize to zero mean and unit variance using training statistics in modeling scenarios.



In [ ]:

scaler = StandardScaler()
X = df[FEATURE_COLS].values
X_scaled = scaler.fit_transform(X)

print('Before standardization (first 5 feature means):', np.round(X.mean(axis=0)[:5], 4))
print('After standardization (first 5 feature means):', np.round(X_scaled.mean(axis=0)[:5], 6))
print('After standardization (first 5 feature stds):', np.round(X_scaled.std(axis=0)[:5], 4))
```

### 2.3 Performing PCA and explained variance

```python
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Robust computation of components for thresholds
n_90 = (np.argmax(cumulative_var >= 0.90) + 1) if (cumulative_var >= 0.90).any() else len(explained_var)
n_95 = (np.argmax(cumulative_var >= 0.95) + 1) if (cumulative_var >= 0.95).any() else len(explained_var)

print('PCA Explained Variance (first 10 components):')
for i, (ev, cv) in enumerate(zip(explained_var[:10], cumulative_var[:10])):
    bar = '█' * int(ev * 50)
    marker = ' ◄── 90%' if i + 1 == n_90 else (' ◄── 95%' if i + 1 == n_95 else '')
    print(f'  PC{i+1:2d}: {ev:6.3f} ({cv:6.3f} cumulative) {bar}{marker}')

print(f'Components for 90% variance: {n_90}')
print(f'Components for 95% variance: {n_95}')

# Scree and cumulative plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(explained_var) + 1), explained_var, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot', fontweight='bold')

axes[1].plot(range(1, len(cumulative_var) + 1), cumulative_var, 'o-', color='steelblue', linewidth=2, markersize=6)
axes[1].axhline(y=0.90, color='red', linestyle='--', linewidth=1.5, label='90% variance')
axes[1].axhline(y=0.95, color='orange', linestyle='--', linewidth=1.5, label='95% variance')
axes[1].axvline(x=n_90, color='red', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance', fontweight='bold')
axes[1].legend()

plt.suptitle('PCA Variance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()




### 2.4 Visualizing data in PCA space (2D)



In [ ]:

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

y = df[TARGET_COL].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='RdYlBu_r', alpha=0.6, s=30, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter1, ax=axes[0], label=TARGET_COL)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title(f'PCA Projection Colored by {TARGET_COL}', fontweight='bold')

sns.kdeplot(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1], fill=True, cmap='Blues', levels=15, ax=axes[1])
axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], alpha=0.2, s=10, color='navy')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
axes[1].set_title('PCA Density', fontweight='bold')

plt.suptitle('Iron Complexes in PCA Space', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pca_2d_projection.png', dpi=150, bbox_inches='tight')
plt.show()




### 2.5 Interpreting PCA loadings (biplot)



In [ ]:

loadings = pd.DataFrame(pca_2d.components_.T, columns=['PC1', 'PC2'], index=FEATURE_COLS)
print('PCA loadings (contribution of each feature to PCs):')
display(loadings)

# Biplot
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='RdYlBu_r', alpha=0.3, s=20)
scale = 3
for i, feature in enumerate(FEATURE_COLS):
    ax.annotate('', xy=(loadings.iloc[i, 0] * scale, loadings.iloc[i, 1] * scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.text(loadings.iloc[i, 0] * scale * 1.15, loadings.iloc[i, 1] * scale * 1.15, feature,
            fontsize=9, fontweight='bold', color='darkred', ha='center', va='center')

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Biplot: Data Points + Feature Loadings', fontsize=13, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretation notes (discuss in mentor checkpoint):
# - Arrows pointing in the same direction indicate positively correlated features.
# - Opposite arrows indicate negative correlation.
# - Longer arrows indicate features with greater influence on the PCs.




---

## Part 3 — K-Means clustering (30 minutes)

### 3.1 Elbow Method and Silhouette analysis

Silhouette coefficient for a sample $i$ is defined as

$$
s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}
$$

where $a(i)$ is the mean intra-cluster distance and $b(i)$ is the mean nearest-cluster distance.



In [ ]:

K_range = range(2, 11)
inertias = []
silhouettes = []
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f'K={k:2d}: WCSS={kmeans.inertia_:10.2f}, Silhouette={silhouettes[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Within-Cluster Sum of Squares (WCSS)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xticks(list(K_range))
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontweight='bold')
axes[1].set_xticks(list(K_range))
axes[1].grid(True, alpha=0.3)

best_k = list(K_range)[int(np.argmax(silhouettes))]
axes[1].axvline(x=best_k, color='green', linestyle='--', linewidth=2, label=f'Best K = {best_k}')
axes[1].legend()

plt.suptitle('Optimal Number of Clusters', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Optimal K by Silhouette Score: {best_k}')




### 3.2 Clustering with optimal K and visualization



In [ ]:

optimal_k = best_k
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df['cluster'] = cluster_labels

# Visualize clusters in PCA space
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
colors = plt.cm.Set1(np.linspace(0, 1, optimal_k))
for k in range(optimal_k):
    mask = cluster_labels == k
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=[colors[k]], label=f'Cluster {k} (n={mask.sum()})',
                    alpha=0.6, s=30, edgecolors='gray', linewidth=0.3)

centroids_pca = pca_2d.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X', s=200, linewidths=2,
                edgecolors='white', zorder=5, label='Centroids')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
axes[0].set_title(f'K-Means Clusters (K={optimal_k}) in PCA Space', fontweight='bold')
axes[0].legend(fontsize=9, loc='best')

# Right: Redox potential distribution per cluster
sns.boxplot(data=df, x='cluster', y=TARGET_COL, palette='Set1', ax=axes[1])
sns.stripplot(data=df, x='cluster', y=TARGET_COL, color='black', alpha=0.3, size=3, jitter=True, ax=axes[1])
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel(TARGET_COL)
axes[1].set_title(f'{TARGET_COL} by Cluster', fontweight='bold')

plt.suptitle('Cluster Analysis of Iron Complexes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()




### 3.3 Cluster characterization



In [ ]:

cluster_stats = df.groupby('cluster')[FEATURE_COLS + [TARGET_COL]].agg(['mean', 'std'])
print('Cluster Characterization:')
for k in range(optimal_k):
    print(f"\nCluster {k} (n = {(cluster_labels == k).sum()} samples):")
    print(f"   {TARGET_COL}: {df[df['cluster']==k][TARGET_COL].mean():.4f} ± {df[df['cluster']==k][TARGET_COL].std():.4f}")
    for feat in FEATURE_COLS[:5]:
        mean_val = df[df['cluster']==k][feat].mean()
        std_val = df[df['cluster']==k][feat].std()
        print(f"   {feat}: {mean_val:.4f} ± {std_val:.4f}")

centroid_df = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=FEATURE_COLS, index=[f'Cluster {k}' for k in range(optimal_k)])
fig, ax = plt.subplots(figsize=(12, max(3, optimal_k)))
sns.heatmap(centroid_df.T, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Cluster Centroids (Original Feature Scale)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_centroids_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()




**Mentor checkpoint 5**

- Confirm how many principal components are needed to capture 90% of the variance for your dataset.
- Discuss which physical features dominate PC1 and PC2 and whether they align with chemical intuition.
- Do the clusters correspond to chemically meaningful groups (ligand types, charge states, etc.)?
- Is there a clear separation in redox potential between clusters?

Proceed only after confirmation.




### Exercise 3.1 — Elbow Method (Required)

1. Run K-Means for K = 2 to 12.
2. Plot both the Elbow curve and Silhouette scores.
3. Justify your choice of optimal K in a Markdown answer.

### Exercise 3.2 — 3D PCA Visualization

1. Fit PCA with 3 components.
2. Create an interactive 3D scatter plot using matplotlib's Axes3D and color points by cluster assignment.
3. Report how much additional variance PC3 captures.

### Exercise 3.3 — t-SNE Comparison (Bonus)

t-SNE optimizes a KL divergence between high- and low-dimensional pairwise distributions:

$$
\mathrm{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}
$$

1. Apply t-SNE with perplexity=30 and compare the 2D embedding to PCA for cluster separation.
2. Discuss why t-SNE may be better for visualization but worse as a preprocessing step for ML.

---

## Summary

| Method | Purpose | Key parameter |
|---|---:|---|
| StandardScaler | Normalize features to zero mean, unit variance | — |
| PCA | Linear dimensionality reduction | n_components |
| K-Means | Partition data into K clusters | n_clusters |
| Elbow Method | Find K by WCSS vs K | WCSS vs K |
| Silhouette Score | Evaluate cluster quality | Range: [-1, 1] |

---

# Notebook 4 — Baseline Machine Learning (Random Forest & Gaussian Process Regression)

**Duration:** 105 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** train and evaluate Random Forest and Gaussian Process Regression baselines, perform basic hyperparameter tuning, and compare predictive performance and uncertainty estimates.

## Learning objectives

- Train Random Forest and Gaussian Process Regression models on engineered features.
- Evaluate models using RMSE, MAE, and R².
- Inspect feature importances from Random Forest and uncertainties from GPR.
- Perform simple hyperparameter tuning and cross-validation.

---

## Part 1 — Theoretical background (25 minutes)

### 1.1 Random Forest Regression

A Random Forest is an ensemble of decision trees trained on bootstrap samples and random feature subsets. At a split, the objective for a candidate split can be written in terms of the mean-squared error (MSE):

$$
\mathrm{MSE}_{\text{split}} = \frac{n_L}{n} \mathrm{MSE}_L + \frac{n_R}{n} \mathrm{MSE}_R
$$

The ensemble prediction is the average of individual tree predictions:

$$
\widehat{y} = \frac{1}{B} \sum_{b=1}^B T_b(\mathbf{x})
$$

Advantages: handles non-linear relationships, robust to outliers, provides feature importance. Disadvantages: cannot extrapolate beyond training data and does not provide uncertainty estimates.

### 1.2 Gaussian Process Regression (GPR)

A Gaussian Process is a distribution over functions specified by a mean function $m(\mathbf{x})$ and a kernel $k(\mathbf{x}, \mathbf{x}')$:

$$
f(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}'))
$$

The Radial Basis Function (RBF) kernel is commonly used:

$$
k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)
$$

Given training data $(X, y)$ and a test point $\mathbf{x}_\ast$, the predictive mean and variance are:

$$
\widehat{y}_\ast = \mathbf{k}_\ast^\top (K + \sigma_n^2 I)^{-1} \mathbf{y}
$$

$$
\mathrm{Var}(\widehat{y}_\ast) = k(\mathbf{x}_\ast, \mathbf{x}_\ast) - \mathbf{k}_\ast^\top (K + \sigma_n^2 I)^{-1} \mathbf{k}_\ast
$$

GPR provides uncertainty estimates but scales as $\mathcal{O}(n^3)$, so it is most practical for small datasets.

### 1.3 Evaluation metrics

Root Mean Squared Error (RMSE):

$$
\mathrm{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \widehat{y}_i)^2}
$$

Coefficient of determination ($R^2$):

$$
R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \widehat{y}_i)^2}{\sum_{i=1}^n (y_i - \overline{y})^2}
$$

Mean Absolute Error (MAE):

$$
\mathrm{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \widehat{y}_i|
$$

---



## Part 2 — Data preparation (15 minutes)

### 2.1 Load and split



In [ ]:

import os
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')

REPO_PATH = os.path.expandvars('$SCRATCH/Fe-Redox-GNN')
df = pd.read_csv(os.path.join(REPO_PATH, 'data_cleaned.csv'))

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
TARGET_COL = numeric_cols[-1]
FEATURE_COLS = [c for c in numeric_cols if c != TARGET_COL]

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

print(f"Dataset: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Target: {TARGET_COL}  Range: [{y.min():.4f}, {y.max():.4f}]  Mean: {y.mean():.4f} ± {y.std():.4f}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# Standardize features (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




### 2.2 Helper functions: evaluation and parity plot


In [ ]:

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def evaluate_model(y_true, y_pred, model_name='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} Performance:\n  RMSE: {rmse:.4f}\n  MAE:  {mae:.4f}\n  R²:   {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'R²': r2}

import matplotlib.pyplot as plt

def parity_plot(y_true, y_pred, model_name='Model', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.5, s=30, edgecolors='gray', linewidth=0.3, color='steelblue')
    lims = [min(np.min(y_true), np.min(y_pred)), max(np.max(y_true), np.max(y_pred))]
    margin = (lims[1] - lims[0]) * 0.05
    lims = [lims[0] - margin, lims[1] + margin]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(f'Actual {TARGET_COL}')
    ax.set_ylabel(f'Predicted {TARGET_COL}')
    ax.set_title(model_name)
    ax.legend()
    ax.set_aspect('equal')
    return ax




---

## Part 3 — Random Forest Regression (25 minutes)

### 3.1 Training a baseline Random Forest



In [ ]:

rf_model = RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_split=2,
                                 min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=-1)
import time
start_time = time.time()
rf_model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time
print(f'Training time: {train_time:.2f} seconds')

# Predictions
y_train_pred_rf = rf_model.predict(X_train_scaled)
y_test_pred_rf = rf_model.predict(X_test_scaled)

train_metrics_rf = evaluate_model(y_train, y_train_pred_rf, 'Random Forest (Train)')
test_metrics_rf = evaluate_model(y_test, y_test_pred_rf, 'Random Forest (Test)')

# Overfitting check
gap = train_metrics_rf['R²'] - test_metrics_rf['R²']
print(f'Overfitting gap (Train R² - Test R²): {gap:.4f}')



### 3.2 Parity plots for Random Forest


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
parity_plot(y_train, y_train_pred_rf, 'Random Forest (Train)', ax=axes[0])
parity_plot(y_test, y_test_pred_rf, 'Random Forest (Test)', ax=axes[1])
plt.suptitle('Random Forest Regression: Parity Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rf_parity_plots.png', dpi=150, bbox_inches='tight')
plt.show()




### 3.3 Feature importance



In [ ]:

importances = rf_model.feature_importances_
importance_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances}).sort_values('Importance', ascending=True)
fig, ax = plt.subplots(figsize=(8, max(4, len(FEATURE_COLS) * 0.4)))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='teal', edgecolor='white', alpha=0.85)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Random Forest Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
for _, row in importance_df.tail(5).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")




---

## Part 4 — Gaussian Process Regression (25 minutes)

### 4.1 Training a GPR model



In [ ]:

from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-3, 1e3)) + WhiteKernel(1e-2)
max_gpr_samples = 1000
if X_train_scaled.shape[0] > max_gpr_samples:
    print(f'Training set ({X_train_scaled.shape[0]}) is large for GPR — subsampling to {max_gpr_samples}')
    rng = np.random.RandomState(42)
    idx = rng.choice(X_train_scaled.shape[0], max_gpr_samples, replace=False)
    X_train_gpr = X_train_scaled[idx]
    y_train_gpr = y_train[idx]
else:
    X_train_gpr = X_train_scaled
    y_train_gpr = y_train

from sklearn.gaussian_process import GaussianProcessRegressor

gpr_model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, alpha=1e-6, random_state=42)
start_time = time.time()
gpr_model.fit(X_train_gpr, y_train_gpr)
train_time = time.time() - start_time
print(f'GPR training time: {train_time:.2f} seconds')
print('Optimized kernel:', gpr_model.kernel_)
print('Log-marginal-likelihood:', gpr_model.log_marginal_likelihood_value_)




### 4.2 Predictions with uncertainty



In [ ]:

y_test_pred_gpr, y_test_std_gpr = gpr_model.predict(X_test_scaled, return_std=True)
y_train_pred_gpr, y_train_std_gpr = gpr_model.predict(X_train_gpr, return_std=True)

train_metrics_gpr = evaluate_model(y_train_gpr, y_train_pred_gpr, 'GPR (Train)')
test_metrics_gpr = evaluate_model(y_test, y_test_pred_gpr, 'GPR (Test)')

# Parity + uncertainty plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
# RF parity
parity_plot(y_test, y_test_pred_rf, 'Random Forest', ax=axes[0])
# GPR parity with error bars
axes[1].errorbar(y_test, y_test_pred_gpr, yerr=2 * y_test_std_gpr, fmt='o', alpha=0.4, markersize=4,
                 color='steelblue', ecolor='lightblue', elinewidth=1, capsize=2)
lims = [min(y_test.min(), y_test_pred_gpr.min()), max(y_test.max(), y_test_pred_gpr.max())]
margin = (lims[1] - lims[0]) * 0.05
lims = [lims[0] - margin, lims[1] + margin]
axes[1].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_xlabel(f'Actual {TARGET_COL}')
axes[1].set_ylabel(f'Predicted {TARGET_COL}')
axes[1].set_title('GPR Predictions with 95% CI', fontweight='bold')
axes[1].legend()
axes[1].set_aspect('equal')

# Uncertainty distribution
axes[2].hist(y_test_std_gpr, bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Predicted Standard Deviation')
axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of Prediction Uncertainty', fontweight='bold')
axes[2].axvline(np.mean(y_test_std_gpr), color='red', linestyle='--', label=f'Mean σ = {np.mean(y_test_std_gpr):.4f}')
axes[2].legend()

plt.suptitle('GPR Results and Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('gpr_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('Uncertainty statistics: mean σ =', np.mean(y_test_std_gpr))




---

## Part 5 — Model comparison (15 minutes)



In [ ]:

results = pd.DataFrame({
    'Model': ['Random Forest', 'GPR'],
    'RMSE (Test)': [test_metrics_rf['RMSE'], test_metrics_gpr['RMSE']],
    'MAE (Test)': [test_metrics_rf['MAE'], test_metrics_gpr['MAE']],
    'R² (Test)': [test_metrics_rf['R²'], test_metrics_gpr['R²']],
    'R² (Train)': [train_metrics_rf['R²'], train_metrics_gpr['R²']],
})

print('Model Comparison Summary:')
display(results)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].bar(results['Model'], results['RMSE (Test)'], color=['steelblue', 'coral'], edgecolor='black', alpha=0.85)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Test RMSE (lower is better)', fontweight='bold')
for i, v in enumerate(results['RMSE (Test)']):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')

axes[1].bar(results['Model'], results['R² (Test)'], color=['steelblue', 'coral'], edgecolor='black', alpha=0.85)
axes[1].set_ylabel('R²')
axes[1].set_title('Test R² (higher is better)', fontweight='bold')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(results['R² (Test)']):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Parity plot for RF on the right
parity_plot(y_test, y_test_pred_rf, 'Random Forest', ax=axes[2])

plt.suptitle('Classical ML Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
results.to_csv(os.path.join(REPO_PATH, 'baseline_results.csv'), index=False)
print('Saved baseline_results.csv for Day 4 comparison')




**Mentor checkpoint 6**

- Which model performed better on the test set and why?
- Is there evidence of overfitting in either model?
- What advantage do GPR uncertainty estimates provide in experimental design?
- What performance improvement would justify using a GNN over these baselines?

Proceed only after confirmation.

---

### Exercise 4.1 — Hyperparameter tuning with GridSearchCV (Required)

1. Define a parameter grid with at least 3 hyperparameters for Random Forest.
2. Use 5-fold cross-validation.
3. Report the best parameters and improvement over the default model.

A starter grid is provided in the notebook as commented code; students should uncomment and run it.

### Exercise 4.2 — Learning curve analysis

1. Train the Random Forest with increasing fractions of the training data (10%, 20%, ..., 100%).
2. Plot train and test RMSE vs. training set size.
3. Discuss whether more data would likely improve performance.

### Exercise 4.3 — Cross-validation analysis

1. Perform 10-fold cross-validation on both models and report the mean and standard deviation of RMSE across folds.
2. Create a box plot comparing CV scores of both models.
3. Discuss whether observed differences are statistically significant.

---

## Summary

| Model | Strengths | Weaknesses | Best For |
|---|---:|---|---|
| Random Forest | Non-linear, feature importance, robust | No uncertainty, can't extrapolate | Medium datasets, feature selection |
| Gaussian Process Regression | Uncertainty estimates, principled | O(n^3) scaling, kernel choice | Small datasets, uncertainty needed |
| GNN (Day 4) | Learns from structure, transferable | Needs more data, complex | Molecular property prediction |

---

## Preparation for Day 4

Day 4 will cover GNNs trained on molecular graphs using PyTorch Geometric on GPU nodes. Before Day 4 ensure:

- `baseline_results.csv` is present in the repository to compare GNN performance against classical baselines.
- You have access to a GPU-enabled kernel (recommend the `fe-redox` conda environment and the kernel named "Python (Fe-Redox)").
- Review the feature shortlist determined in Mentor checkpoint 6; these features will be used as baselines and for ablation studies.

